In [14]:
import os
from pathlib import Path

PROJECT_PATH = Path(
    "/Users/ilyaklimasin/ecommerce_product_analytics"
).resolve()

os.chdir(PROJECT_PATH)

print("Рабочая папка:", Path.cwd())
print(
    "CSV найден:",
    Path("data/raw/olist_customers_dataset.csv").exists()
)

Рабочая папка: /Users/ilyaklimasin/ecommerce_product_analytics
CSV найден: True


In [8]:
DATABASE_PATH = PROJECT_PATH / "ecommerce.duckdb"

connection = duckdb.connect(str(DATABASE_PATH))

print("Подключение создано")
print("База данных:", DATABASE_PATH)

Подключение создано
База данных: /Users/ilyaklimasin/ecommerce_product_analytics/ecommerce.duckdb


In [15]:
def execute_sql_file(sql_file_name):
    sql_path = PROJECT_PATH / "sql" / sql_file_name
    sql_code = sql_path.read_text(encoding="utf-8")
    
    connection.execute(sql_code)
    
    print(f"Файл {sql_file_name} выполнен")

execute_sql_file("01_create_tables.sql")

Файл 01_create_tables.sql выполнен


In [16]:
connection.sql("SHOW TABLES").df()

,name
0,category_translation
1,customers
2,geolocation
3,items_by_order
4,order_items
5,orders
6,orders_enriched_sql
7,payments
8,payments_by_order
9,products


In [17]:
connection.sql("""
    SELECT *
    FROM orders_enriched_sql
    LIMIT 5
""").df()

,order_id,customer_id,customer_unique_id,customer_city,customer_state,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,payment_records,installments_max,payment_types_count,review_score,reviews_count,purchase_month,delivery_days,delay_days,is_late,is_canceled
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,...,3,1,2,4.0,1,2017-10-01,8.436574,-7.107488,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,af07308b275d755c9edb36a90c618231,barreiras,BA,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,...,1,1,1,4.0,1,2018-07-01,13.782037,-5.355729,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,...,1,3,1,5.0,1,2018-08-01,9.394213,-17.245498,False,False
3,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,...,1,1,1,5.0,1,2018-02-01,2.873877,-9.238171,False,False
4,a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,80bb27c7c16e8f973207a5086ab329e2,congonhinhas,PR,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,...,1,6,1,4.0,1,2017-07-01,16.542245,-5.543113,False,False


In [18]:
connection.sql("""
    SELECT
        COUNT(*) AS rows_count,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders_enriched_sql
""").df()

,rows_count,unique_orders
0,99441,99441


In [19]:
execute_sql_file("02_business_metrics.sql")

Файл 02_business_metrics.sql выполнен


In [20]:
connection.sql("""
    SELECT *
    FROM kpi_summary_sql
""").df()

,delivered_orders,unique_customers,product_revenue,average_order_value,average_items_per_order,average_delivery_days,late_delivery_rate_percent,average_review_score,cancellation_rate_percent
0,96478,93358,13221498.11,137.04,1.14,12.56,8.11,4.156,0.63


In [21]:
connection.sql("""
    SELECT *
    FROM monthly_metrics_sql
    ORDER BY purchase_month
    LIMIT 10
""").df()

,purchase_month,orders_count,unique_customers,product_revenue,average_order_value,average_delivery_days,late_delivery_rate_percent,average_review_score
0,2017-01-01,750,718,111798.36,149.06,12.65,3.07,4.199
1,2017-02-01,1653,1630,234223.40,141.70,13.17,3.21,4.203
2,2017-03-01,2546,2508,359198.85,141.08,12.95,5.58,4.187
3,2017-04-01,2303,2274,340669.68,147.92,14.92,7.86,4.138
4,2017-05-01,3546,3479,489338.25,138.00,11.32,3.61,4.238
5,2017-06-01,3135,3076,421923.37,134.58,12.01,3.86,4.223
6,2017-07-01,3872,3802,481604.52,124.38,11.59,3.43,4.258
7,2017-08-01,4193,4114,554699.70,132.29,11.15,3.32,4.312
8,2017-09-01,4150,4083,607399.67,146.36,11.85,5.20,4.267
9,2017-10-01,4478,4417,648247.65,144.76,11.86,5.29,4.205


In [22]:
connection.sql("""
    SELECT *
    FROM delivery_group_summary_sql
    ORDER BY delivery_group
""").df()

,delivery_group,group_order,orders_count,average_review,low_review_rate_percent
0,0-7,1,25933,4.418,7.44
1,15-21,3,17582,4.140,11.60
2,22-30,4,7882,3.607,24.28
3,31+,5,4430,2.254,62.91
4,8-14,2,39997,4.312,8.92


In [24]:
execute_sql_file("03_customer_analysis.sql")


connection.sql("""
    SELECT *
    FROM segment_summary_sql
    ORDER BY customers_count DESC
""").df()

Файл 03_customer_analysis.sql выполнен


,segment,customers_count,average_recency,average_frequency,average_monetary,total_monetary,customers_share_percent
0,Regular one-time,28810,213.11,1.00,61.87,1782403.21,30.86
1,New customers,22554,59.89,1.00,140.04,3158460.45,24.16
2,At risk,17199,397.22,1.04,199.04,3423242.69,18.42
3,Inactive one-time,11780,453.22,1.00,46.67,549759.52,12.62
4,High-value one-time,10795,224.36,1.00,344.85,3722648.46,11.56
5,Champions,1310,115.34,2.16,300.23,393299.50,1.40
6,Loyal customers,910,237.69,2.06,210.64,191684.28,0.97


In [25]:
connection.sql("""
    SELECT *
    FROM repeat_customer_summary_sql
""").df()

,customers_count,repeat_customers,repeat_customer_rate_percent
0,93358,2801.0,3.0


In [27]:
connection.sql("""
    SELECT *
    FROM cohort_retention_long_sql
    ORDER BY cohort_month, cohort_index
    LIMIT 20
""").df()

,cohort_month,cohort_index,customers_count,cohort_size,retention_rate
0,2017-01-01,0,717,717,1.00000
1,2017-01-01,1,2,717,0.00279
2,2017-01-01,2,2,717,0.00279
3,2017-01-01,3,1,717,0.00139
4,2017-01-01,4,3,717,0.00418
5,2017-01-01,5,1,717,0.00139
6,2017-01-01,6,3,717,0.00418
7,2017-01-01,7,1,717,0.00139
8,2017-01-01,8,1,717,0.00139
9,2017-01-01,10,3,717,0.00418


In [28]:
connection.sql("SHOW TABLES").df()

,name
0,category_translation
1,cohort_retention_long_sql
2,customer_frequency_sql
3,customer_rfm_scores_sql
4,customer_rfm_sql
5,customer_segments_sql
6,customers
7,delivery_group_summary_sql
8,delivery_status_summary_sql
9,financial_check_sql


In [29]:
connection.sql("""
    SELECT
        COUNT(*) AS delivered_orders,
        COUNT(DISTINCT customer_unique_id) AS unique_customers,
        ROUND(SUM(product_revenue), 2) AS product_revenue,
        ROUND(AVG(product_revenue), 2) AS average_order_value
    FROM orders_enriched_sql
    WHERE order_status = 'delivered'
""").df()

,delivered_orders,unique_customers,product_revenue,average_order_value
0,96478,93358,13221498.11,137.04


In [30]:
connection.close()